# Iteración V2: Double DQN

En esta iteración se implementa Double DQN utilizando la misma arquitectura convolucional y los mismos hiperparámetros principales del DQN vanilla.

En DQN vanilla, la red objetivo selecciona y evalúa la acción futura mediante:

$$
y = r + \gamma(1-d)\max_{a'}Q_{\text{target}}(s',a')
$$

Double DQN separa ambas responsabilidades:

1. La red online selecciona la acción con mayor valor estimado:

$$
a^* = \arg\max_{a'}Q_{\text{online}}(s',a')
$$

2. La red target evalúa la acción seleccionada:

$$
y = r + \gamma(1-d)Q_{\text{target}}(s',a^*)
$$

Esta modificación no cambia la arquitectura de la CNN. Solamente cambia la manera en que se calcula el target durante el entrenamiento.

El objetivo de esta iteración es determinar si Double DQN:

- Reduce la sobreestimación de los valores Q.
- Produce evaluaciones más estables.
- Supera el promedio de 406.67 puntos obtenido por DQN vanilla.
- Obtiene un mejor resultado bajo el criterio de competencia.

In [1]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

import models
import replay_buffer
import train
import evaluation

# Recargar los módulos por si fueron modificados
importlib.reload(models)
importlib.reload(replay_buffer)
importlib.reload(train)
importlib.reload(evaluation)

from models import DQN
from replay_buffer import ReplayBuffer
from train import (
    ConfigDQN,
    actualizar_modelo,
    entrenar_dqn,
)
from evaluation import evaluar_modelo

SEMILLA = 42
N_ACCIONES = 6

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo seleccionado: {device}")
print(f"Número de acciones: {N_ACCIONES}")
print(f"Semilla: {SEMILLA}")

Dispositivo seleccionado: mps
Número de acciones: 6
Semilla: 42


## Probar la actualización Double DQN

In [2]:
from copy import deepcopy

config_prueba = ConfigDQN(
    batch_size=8,
    capacidad_buffer=16,
)

modelo_online_prueba = DQN(
    n_acciones=N_ACCIONES
).to(device)

modelo_target_prueba = deepcopy(
    modelo_online_prueba
).to(device)

modelo_target_prueba.eval()

buffer_prueba = ReplayBuffer(
    capacidad=16,
    forma_observacion=(4, 84, 84),
    seed=SEMILLA,
)

rng_prueba = np.random.default_rng(SEMILLA)

for _ in range(16):
    observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    siguiente_observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    buffer_prueba.agregar(
        observacion=observacion,
        accion=int(rng_prueba.integers(N_ACCIONES)),
        recompensa=float(
            rng_prueba.choice([-1.0, 0.0, 1.0])
        ),
        siguiente_observacion=siguiente_observacion,
        finalizado=bool(rng_prueba.random() < 0.1),
    )

optimizador_prueba = torch.optim.Adam(
    modelo_online_prueba.parameters(),
    lr=config_prueba.learning_rate,
)

pesos_antes = {
    nombre: parametro.detach().clone()
    for nombre, parametro
    in modelo_online_prueba.named_parameters()
}

metricas_prueba = actualizar_modelo(
    modelo_online=modelo_online_prueba,
    modelo_target=modelo_target_prueba,
    replay_buffer=buffer_prueba,
    optimizador=optimizador_prueba,
    config=config_prueba,
    device=device,
    usar_double_dqn=True,
)

parametros_modificados = sum(
    not torch.equal(
        pesos_antes[nombre],
        parametro.detach(),
    )
    for nombre, parametro
    in modelo_online_prueba.named_parameters()
)

print("PRUEBA DOUBLE DQN")
print(f"Loss: {metricas_prueba['loss']:.6f}")
print(
    f"Valor Q promedio: "
    f"{metricas_prueba['q_promedio']:.6f}"
)
print(
    f"Target promedio: "
    f"{metricas_prueba['target_promedio']:.6f}"
)
print(
    f"Tensores modificados: "
    f"{parametros_modificados}"
)

assert np.isfinite(metricas_prueba["loss"])
assert parametros_modificados > 0

print("\nLa actualización Double DQN funciona correctamente.")

PRUEBA DOUBLE DQN
Loss: 0.273538
Valor Q promedio: -0.003942
Target promedio: 0.300140
Tensores modificados: 10

La actualización Double DQN funciona correctamente.
